# ChessMind Arena — Chess Policy Model Training

This notebook trains the first chess policy network for ChessMind Arena.

The model learns:

    Chess Position → Strong Candidate Move

The training data comes from the prepared Hugging Face dataset:

    chessmind-arena-policy-dataset

Each example contains:

- FEN
- Stockfish principal-variation first move
- Stockfish evaluation
- search depth
- node count

The first model is a CNN-based policy network.

The model is not intended to replace the LLM.

Instead, it will eventually be used as:

    Chess Position
          ↓
    Chess Policy Network
          ↓
    Top-K Candidate Moves
          ↓
    LLM Player
          ↓
    Selected Move

This notebook establishes the first supervised-learning baseline.

In [1]:
!pip install -q datasets python-chess torch pandas numpy tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 79.0 MB/s eta 0:00:00:00:010:01
  Preparing metadata (setup.py) ... done


In [33]:
import chess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from copy import deepcopy
import getpass
from huggingface_hub import HfApi
import os
import json
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
SEED = 42
BATCH_SIZE = 256
learning_rate = 1e-3
EPOCHS = 10

NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [ ]:
os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face Token: ")

In [5]:
HF_DATASET_ID = "Mridul-Dixit/chessmind-arena-policy-dataset"

dataset = load_dataset(HF_DATASET_ID, token=os.environ["HF_TOKEN"])

print(dataset)

README.md:   0%|          | 0.00/668 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.03MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  829kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/160000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 160000
    })
    validation: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['fen', 'target_move', 'cp', 'mate', 'depth', 'knodes'],
        num_rows: 20000
    })
})


In [6]:
sample = dataset["train"][0]

In [7]:
print("FEN:", sample["fen"])
print("Target move:", sample["target_move"])
print("Depth:", sample["depth"])
print("CP:", sample["cp"])
print("Mate:", sample["mate"])

FEN: 6k1/p7/1pKRr2p/2n1P3/8/P7/8/3N4 w - -
Target move: d6e6
Depth: 48
CP: 0.0
Mate: None


In [8]:
board = chess.Board(sample["fen"])

print(board)

. . . . . . k .
p . . . . . . .
. p K R r . . p
. . n . P . . .
. . . . . . . .
P . . . . . . .
. . . . . . . .
. . . N . . . .


In [9]:
print("\nTarget move:", sample["target_move"])
move = chess.Move.from_uci(sample["target_move"])
print("Is legal:", move in board.legal_moves)


Target move: d6e6
Is legal: True


In [10]:

PIECE_TO_CHANNEL = {
    (chess.WHITE, chess.PAWN): 0,
    (chess.WHITE, chess.KNIGHT): 1,
    (chess.WHITE, chess.BISHOP): 2,
    (chess.WHITE, chess.ROOK): 3,
    (chess.WHITE, chess.QUEEN): 4,
    (chess.WHITE, chess.KING): 5,
    (chess.BLACK, chess.PAWN): 6,
    (chess.BLACK, chess.KNIGHT): 7,
    (chess.BLACK, chess.BISHOP): 8,
    (chess.BLACK, chess.ROOK): 9,
    (chess.BLACK, chess.QUEEN): 10,
    (chess.BLACK, chess.KING): 11,
}

INPUT_CHANNELS = 18


def encode_board(fen):
    board = chess.Board(fen)

    encoded = np.zeros(
        (INPUT_CHANNELS, 8, 8),
        dtype=np.float32
    )

    # Channels 0–11: piece locations
    for square, piece in board.piece_map().items():
        channel = PIECE_TO_CHANNEL[
            (piece.color, piece.piece_type)
        ]
        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)
        encoded[channel, row, col] = 1.0

    # Channel 12: side to move
    if board.turn == chess.BLACK:
        encoded[12, :, :] = 1.0

    # Channels 13–16: castling rights
    castling_checks = [
        board.has_kingside_castling_rights(chess.WHITE),
        board.has_queenside_castling_rights(chess.WHITE),
        board.has_kingside_castling_rights(chess.BLACK),
        board.has_queenside_castling_rights(chess.BLACK),
    ]

    for channel, has_right in enumerate(castling_checks, start=13):
        if has_right:
            encoded[channel, :, :] = 1.0

    # Channel 17: en-passant target square
    if board.ep_square is not None:
        row = 7 - chess.square_rank(board.ep_square)
        col = chess.square_file(board.ep_square)
        encoded[17, row, col] = 1.0

    return encoded


In [11]:
encoded_board = encode_board(sample["fen"])

print("Encoded board shape:", encoded_board.shape)
print("Non zero entries:", np.count_nonzero(encoded_board))

Encoded board shape: (18, 8, 8)
Non zero entries: 11


In [12]:
NUM_SQUARES = 64
BASE_MOVE_CLASSES = NUM_SQUARES * NUM_SQUARES
PROMOTION_OFFSET = BASE_MOVE_CLASSES

PROMOTION_PIECES = [
    chess.KNIGHT,
    chess.BISHOP,
    chess.ROOK,
    chess.QUEEN,
]

PROMOTION_TO_INDEX = {
    chess.KNIGHT: 0,
    chess.BISHOP: 1,
    chess.ROOK: 2,
    chess.QUEEN: 3,
}

NUM_MOVE_CLASSES = BASE_MOVE_CLASSES * (
    1 + len(PROMOTION_PIECES)
)


def encode_move(uci_move):
    move = chess.Move.from_uci(uci_move)

    base_index = (
        move.from_square * NUM_SQUARES
        + move.to_square
    )

    if move.promotion is None:
        return base_index

    if move.promotion not in PROMOTION_TO_INDEX:
        raise ValueError(
            f"Unsupported promotion piece: {uci_move}"
        )

    promotion_index = PROMOTION_TO_INDEX[move.promotion]

    return (
        PROMOTION_OFFSET
        + promotion_index * BASE_MOVE_CLASSES
        + base_index
    )


def decode_move(move_index):
    if not 0 <= move_index < NUM_MOVE_CLASSES:
        raise ValueError(f"Invalid move index: {move_index}")

    if move_index < BASE_MOVE_CLASSES:
        base_index = move_index
        promotion_piece = None
    else:
        promotion_index = (
            move_index - PROMOTION_OFFSET
        ) // BASE_MOVE_CLASSES

        base_index = (
            move_index - PROMOTION_OFFSET
        ) % BASE_MOVE_CLASSES

        promotion_piece = PROMOTION_PIECES[promotion_index]

    from_square = base_index // NUM_SQUARES
    to_square = base_index % NUM_SQUARES

    return chess.Move(
        from_square,
        to_square,
        promotion=promotion_piece
    )


In [13]:

test_moves = [
    "e8a8",  # Castling UCI representation in this dataset
    "e2e4",  # Ordinary move
    "e7e8n", # Knight promotion
    "e7e8b", # Bishop promotion
    "e7e8r", # Rook promotion
    "e7e8q", # Queen promotion
]

for uci in test_moves:
    encoded = encode_move(uci)
    decoded = decode_move(encoded).uci()

    print(uci, "->", encoded, "->", decoded)
    assert decoded == uci

print("Move encoding round-trip tests passed.")


e8a8 -> 3896 -> e8a8
e2e4 -> 796 -> e2e4
e7e8n -> 7484 -> e7e8n
e7e8b -> 11580 -> e7e8b
e7e8r -> 15676 -> e7e8r
e7e8q -> 19772 -> e7e8q
Move encoding round-trip tests passed.


In [14]:

class ChessPolicyDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data[index]
        fen = row["fen"]
        target_move = row["target_move"]

        board = chess.Board(fen)

        # Encode the board and target.
        encoded = encode_board(fen)
        target_index = encode_move(target_move)

        # True means that the move class is legal.
        legal_mask = torch.zeros(
            NUM_MOVE_CLASSES,
            dtype=torch.bool
        )

        legal_moves = list(board.legal_moves)

        if not legal_moves:
            raise ValueError(
                f"Terminal position unexpectedly present: {fen}"
            )

        legal_indices = [
            encode_move(move.uci())
            for move in legal_moves
        ]
        legal_mask[legal_indices] = True

        # The training target must be legal.
        if not legal_mask[target_index]:
            raise ValueError(
                f"Target {target_move} is illegal for FEN: {fen}"
            )

        return (
            torch.tensor(encoded, dtype=torch.float32),
            torch.tensor(target_index, dtype=torch.long),
            legal_mask
        )


In [15]:
train_dataset = ChessPolicyDataset(
    dataset["train"]
)
val_dataset = ChessPolicyDataset(
    dataset["validation"]
)
test_dataset = ChessPolicyDataset(
    dataset["test"]
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 160000
Validation dataset size: 20000
Test dataset size: 20000


In [16]:
X, y, _  = train_dataset[0]

print("Input shape:", X.shape)
print("Target:", y.item())
print("Target move:", dataset["train"][0]["target_move"])


Input shape: torch.Size([18, 8, 8])
Target: 2796
Target move: d6e6


In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 625
Validation batches: 79
Test batches: 79


In [18]:
class ChessPolicyNet(nn.Module):
    def __init__(self, input_channels=12, num_classes=NUM_MOVE_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.policy_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.policy_head(x)
        return x

In [19]:
model = ChessPolicyNet(input_channels=INPUT_CHANNELS, num_classes=NUM_MOVE_CLASSES).to(DEVICE)

print(model)

ChessPolicyNet(
  (features): Sequential(
    (0): Conv2d(18, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
  )
  (policy_head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=20480, bias=True)
  )
)


In [20]:
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Total parameters: 12909568


In [21]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-3
)   

In [22]:
def calculate_top_k_accuracy(
    logits,
    targets,
    k
):
    _, predictions = torch.topk(
        logits,
        k=k,
        dim=1
    )

    correct = (
        predictions == targets.unsqueeze(1)
    ).any(dim=1)

    return correct.float().mean().item()

In [23]:

def apply_legal_move_mask(logits, legal_masks):
    """
    logits:      [batch_size, NUM_MOVE_CLASSES]
    legal_masks: [batch_size, NUM_MOVE_CLASSES]
                 True = legal, False = illegal
    """
    legal_masks = legal_masks.to(
        device=logits.device,
        dtype=torch.bool
    )

    if not legal_masks.any(dim=1).all():
        raise ValueError("A position has no legal moves.")

    masked_logits = logits.masked_fill(
        ~legal_masks,
        float("-inf")
    )

    return masked_logits


In [24]:

def evaluate_model(model, loader):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    hits = {1: 0, 3: 0, 5: 0, 10: 0}

    with torch.no_grad():
        for inputs, targets, legal_masks in loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            legal_masks = legal_masks.to(DEVICE)

            logits = model(inputs)
            masked_logits = apply_legal_move_mask(
                logits, legal_masks
            )

            loss = criterion(masked_logits, targets)

            batch_size = inputs.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            for k in hits:
                predictions = torch.topk(
                    masked_logits, k=k, dim=1
                ).indices

                correct = (
                    predictions == targets.unsqueeze(1)
                ).any(dim=1)

                hits[k] += correct.sum().item()

    return {
        "loss": total_loss / total_samples,
        "top1": hits[1] / total_samples,
        "top3": hits[3] / total_samples,
        "top5": hits[5] / total_samples,
        "top10": hits[10] / total_samples,
    }


In [26]:
PATIENCE = 3
MIN_DELTA = 0.001

best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0
history = []


for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    total_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for inputs, targets, legal_masks in progress:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)
        legal_masks = legal_masks.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(inputs)
        masked_logits = apply_legal_move_mask(
            logits, legal_masks
        )

        loss = criterion(masked_logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = running_loss / total_samples
    val_metrics = evaluate_model(model, val_loader)

    print(
        f"Epoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_metrics['loss']:.4f}, "
        f"val_top1={val_metrics['top1']:.2%}, "
        f"val_top5={val_metrics['top5']:.2%}, "
        f"val_top10={val_metrics['top10']:.2%}"
    )
    current_val_loss = val_metrics["loss"]

    if current_val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = current_val_loss
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0

        torch.save(
            best_state,
            "best_chess_policy.pt"
        )

        print("Saved improved checkpoint.")

    else:
        epochs_without_improvement += 1
        print(
            "No validation improvement:",
            epochs_without_improvement,
            "/",
            PATIENCE
        )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

    epoch_result = {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                **{
                    f"val_{k}": v
                    for k, v in val_metrics.items()
                }
            }
     
    history.append(epoch_result)

Epoch 1/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 1: train_loss=2.7332, val_loss=2.7529, val_top1=15.56%, val_top5=48.02%, val_top10=67.66%
Saved improved checkpoint.


Epoch 2/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 2: train_loss=2.7309, val_loss=2.7518, val_top1=15.46%, val_top5=47.96%, val_top10=67.65%
Saved improved checkpoint.


Epoch 3/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 3: train_loss=2.7292, val_loss=2.7534, val_top1=15.65%, val_top5=48.08%, val_top10=67.55%
No validation improvement: 1 / 3


Epoch 4/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 4: train_loss=2.7270, val_loss=2.7515, val_top1=15.65%, val_top5=48.16%, val_top10=67.77%
No validation improvement: 2 / 3


Epoch 5/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 5: train_loss=2.7246, val_loss=2.7517, val_top1=15.60%, val_top5=48.15%, val_top10=67.72%
No validation improvement: 3 / 3
Early stopping triggered.


In [27]:
if best_state is not None:
    model.load_state_dict(best_state)

print("Best validation loss:", best_val_loss)

Best validation loss: 2.751794504547119


In [28]:
test_metrics = evaluate_model(
    model,
    test_loader
)

print("Test Results")
print("------------")

for metric, value in test_metrics.items():

    print(f"{metric}: {value:.4f}")

Test Results
------------
loss: 2.7540
top1: 0.1558
top3: 0.3573
top5: 0.4818
top10: 0.6745


In [29]:
def get_candidate_moves(
    model,
    fen,
    top_k=5
):

    model.eval()

    board = chess.Board(fen)

    encoded = encode_board(fen)

    tensor = torch.tensor(
        encoded,
        dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        logits = model(tensor)[0]

    legal_moves = list(
        board.legal_moves
    )

    legal_indices = []

    for move in legal_moves:

        move_index = encode_move(
            move.uci()
        )

        legal_indices.append(
            (move_index, move)
        )

    scored_moves = []

    for move_index, move in legal_indices:

        score = logits[move_index].item()

        scored_moves.append(
            (move.uci(), score)
        )

    scored_moves.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return scored_moves[:top_k]

In [30]:
def inspect_prediction(model, row, top_k=5):
    model.eval()

    board = chess.Board(row["fen"])
    target = chess.Move.from_uci(row["target_move"])

    assert target in board.legal_moves, (
        f"Illegal target: {row['target_move']}"
    )

    candidates = get_candidate_moves(
        model,
        row["fen"],
        top_k=top_k
    )

    candidate_moves = [move for move, score in candidates]

    print(board)
    print("\nTarget move:", target.uci())
    print("Target is legal:", target in board.legal_moves)
    print("Target in top-k:", target.uci() in candidate_moves)

    print("\nTop candidates:")
    for rank, (move, score) in enumerate(candidates, 1):
        print(f"{rank}. {move} (score={score:.4f})")

    return target.uci() in candidate_moves

In [31]:
for i in range(5):
    print(f"\n{'=' * 15} Test position {i + 1} {'=' * 15}")
    inspect_prediction(model, dataset["test"][i], top_k=5)


=============== Test position 1 ===============
r . r . . . k .
. p . . p p b p
. . . p b n p .
. N . . . . B .
. . P . P . . .
. P . . . P . .
. . . R B . P P
. R . . . . K .

Target move: g2g3
Target is legal: True
Target in top-k: False

Top candidates:
1. g5f6 (score=1.3464)
2. b5d6 (score=0.7946)
3. e4e5 (score=0.5362)
4. b5d4 (score=0.4356)
5. g5e3 (score=0.3884)

=============== Test position 2 ===============
. . . . . . . .
. . r . . p k .
. . . . b n . .
p . . p . . . .
. r . . . . p .
. . N R P . P .
. P . R . P B K
. . . . . . . .

Target move: g2d5
Target is legal: True
Target in top-k: False

Top candidates:
1. c3e4 (score=0.4075)
2. c3d5 (score=0.0891)
3. e3e4 (score=-0.0750)
4. g2f3 (score=-0.1480)
5. d3d4 (score=-0.1565)

=============== Test position 3 ===============
. . . . . . . .
. . . . . b . .
. . . . . . . .
p . . . . . . .
. . . . . . . .
k . . R P . . .
P . . K . . . .
. . . . . . . .

Target move: a3b4
Target is legal: True
Target in top-k: True

Top candid

In [32]:
test_fen = dataset["test"][0]["fen"]

print(
    chess.Board(test_fen)
)

print(
    "\nStockfish target:",
    dataset["test"][0]["target_move"]
)

candidates = get_candidate_moves(
    model,
    test_fen,
    top_k=5
)

print("\nPolicy candidates:")

for move, score in candidates:

    print(
        f"{move:6s} {score:.4f}"
    )

r . r . . . k .
. p . . p p b p
. . . p b n p .
. N . . . . B .
. . P . P . . .
. P . . . P . .
. . . R B . P P
. R . . . . K .

Stockfish target: g2g3

Policy candidates:
g5f6   1.3464
b5d6   0.7946
e4e5   0.5362
b5d4   0.4356
g5e3   0.3884


In [34]:
# Use the best validation checkpoint if your training loop saved one.
if "best_model_state" in globals() and best_state is not None:
    model.load_state_dict(best_state)

model.eval()

MODEL_DIR = Path("chessmind_policy_checkpoint")
MODEL_DIR.mkdir(exist_ok=True)

# Save tensors on CPU so the checkpoint can load on CPU or GPU.
state_dict = {
    key: value.detach().cpu()
    for key, value in model.state_dict().items()
}

checkpoint = {
    "model_state_dict": state_dict,
    "config": {
        "input_channels": 18,
        "num_classes": 20480,
        "best_validation_loss": 2.7517,
        "architecture": "ChessPolicyNet",
        "move_encoding": "from_square_to_square_with_promotion",
        "board_encoding": "18_channel",
    },
}

torch.save(checkpoint, MODEL_DIR / "model_checkpoint.pth")

with open(MODEL_DIR / "config.json", "w") as f:
    json.dump(checkpoint["config"], f, indent=2)

print("Saved files:")
for path in MODEL_DIR.iterdir():
    print(path, f"({path.stat().st_size / 1024 / 1024:.2f} MB)")

Saved files:
chessmind_policy_checkpoint/config.json (0.00 MB)
chessmind_policy_checkpoint/model_checkpoint.pth (49.25 MB)


In [36]:
REPO_ID = "Mridul-Dixit/chessmind-arena-policy-model"

api = HfApi(token=os.environ["HF_TOKEN"])

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,
)

api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(MODEL_DIR),
    commit_message="Upload trained ChessMind Arena policy network",
)

print(f"Uploaded checkpoint to https://huggingface.co/{REPO_ID}")

Uploaded checkpoint to https://huggingface.co/Mridul-Dixit/chessmind-arena-policy-model
